> 本章结束时，你闭着眼都能画出 `(B, L, D)` 到 `(B, H, L, Dh)` 的形状芭蕾。

---

## 第29章　注意力核心：QKV与缩放点积

> 本章目标：手写单头自注意力，彻底搞懂 `Q @ Kᵀ / √d` 的物理意义。不碰多头，不碰 FFN。
> 前置知识：第 5 章（点积）、第 6 章（矩阵乘法）、第 22 章（softmax）

> 🕵️ **开篇导语：一场"信息审讯"**
>
> 想象你是一位侦探，正坐在审讯室里，面对一排嫌疑人（**输入序列**）。此刻，你的目光紧紧盯着**当前**这个人（**Query**）——你心里有一个明确的问题："昨晚10点你在哪儿？"
>
> 你迅速扫视屋内其他所有人，他们每人胸前都挂着一张"线索牌"（**Keys**）：有人写"我在酒吧"，有人写"我在家"，还有人写"我看见了可疑车辆"。你的大脑飞速计算：谁的牌子和我的问题最匹配（**Q·K 点积**）？匹配度越高，你就越应该集中精力听取**这个人**的完整证词（**Values**）；匹配度太低的人，你直接无视（Softmax 后权重趋近于 0）。这就是**缩放点积注意力**最底层的物理直觉。
>
> 但仅凭"语义匹配"还不够——嫌疑人的供词必须遵循时间顺序（你不可能让昨晚10点发生的事，被今早8点的证词提前透露）。这就是**因果掩码（Causal Mask）**的职责：屏蔽未来信息。同时，为了让模型感知"第3个词"和"第8个词"的相对位置，我们还必须给每个嫌疑人贴上一张**位置标签**（Sinusoidal、Learned，或当前最流行的 **RoPE 旋转位置编码**）。
>
> 本章，我们不碰多头，不碰 FFN——只专注一件事：把这场"信息审讯"逐行翻译成 NumPy/PyTorch 张量代码。当你读完本章，`Q @ K.T / sqrt(d_k)` 在你眼中将不再是一堆抽象符号，而是嫌疑人胸前那张正在闪烁的"匹配度评分表"。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("env ready")



### 29.1　从词向量到嵌入 —— Token → 查表 → 向量

Transformer 无法直接处理文字。第一步是把每个 token（词或子词）映射为一个稠密向量——这叫**嵌入（Embedding）**。本质上是一次查表操作：词表大小 V×嵌入维度 d_model 的矩阵中，取第 token_id 行。

输入文本经过 Tokenizer 变成整数 ID 序列，再经过 Embedding 层变成 `(batch, seq_len, d_model)` 的张量。这个张量就是后续所有计算的起点——第 4 章向量的语义运算、第 5 章的点积相似度、第 6 章矩阵乘法——全都作用在这个张量上。

📐 **定义　Token Embedding**：查表操作，将离散 token ID 映射为连续向量。`X = Embedding[input_ids]`，shape = (batch, seq_len, d_model)。

💻 **代码　模拟 Embedding 层**




In [ ]:
import numpy as np

# 词表大小 1000，嵌入维度 512
vocab_size, d_model = 1000, 512
embedding = np.random.randn(vocab, d_model) * 0.02  # 模拟预训练嵌入

# 模拟一条输入："我 爱 机器 学习" → 4 个 token
input_ids = np.array([42, 108, 356, 901])
X = embedding[input_ids]  # (4, 512) —— 查表！
print(f"输入 token IDs: {input_ids}")
print(f"嵌入后 shape: {X.shape} ← (seq_len=4, d_model=512)")




---

### 29.2　位置编码的三种直觉 —— 让模型感知"第几个词"

嵌入层有一个致命缺陷：查表操作**不知道 token 的位置**。"我爱机器学习"和"学习机器爱我"——对嵌入层来说只是两组相同的向量，顺序信息完全丢失。

**位置编码（Positional Encoding）** 的使命：给每个位置的嵌入向量加上一个独特的"位置标签"，让模型能区分第 1 个词和第 4 个词。

三种方案：

- **Sinusoidal（原始 Transformer）**：用 sin/cos 函数生成固定的位置编码——不同频率的正弦波在不同位置产生不同值。无需额外参数，但对相对位置的表达较弱。
- **Learned（BERT）**：把位置编码也当成可学习的 Embedding——模型自己学。优点是灵活，缺点是只能处理训练时见过的最大长度。
- **RoPE（LLaMA/Qwen）**：旋转位置编码——通过旋转向量来编码位置。核心洞察：旋转操作天然保持了两个位置之间的**相对角度**不变。RoPE 能更好捕捉相对位置关系，且可外推到更长序列。

📐 **定义　位置编码**：PE(pos, 2i) = sin(pos/10000^(2i/d))，PE(pos, 2i+1) = cos(pos/10000^(2i/d))。加到 token embedding 上。`X = Embedding + PE`。

💻 **代码　Sinusoidal 位置编码 + 可视化**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_pe(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (2 * i / d_model)
            pe[pos, i] = np.sin(pos / denom)
            pe[pos, i+1] = np.cos(pos / denom)
    return pe

seq_len, d_model = 50, 128
pe = sinusoidal_pe(seq_len, d_model)

# 可视化前 4 个维度的位置编码
plt.figure(figsize=(10, 4))
for dim in [0, 1, 30, 31]:
    plt.plot(range(seq_len), pe[:, dim], label=f'dim {dim}')
plt.xlabel('Position'); plt.ylabel('Encoding value')
plt.title('Sinusoidal Positional Encoding (4 dimensions)')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("每个位置有独特的编码模式——模型由此感知词序")




---

### 29.3　QKV 三兄弟的角色扮演 —— 核心类比

现在 X 既有语义信息（Embedding）又有位置信息（PE）。接下来是最关键的步骤：**将 X 投影为 Q、K、V 三个矩阵。**

- **Q（Query）**："我要找什么？"——当前 token 的需求表达
- **K（Key）**："我有什么？"——所有 token 的标签/索引
- **V（Value）**："找到后取出什么？"——所有 token 的实际内容

三者通过三个不同的权重矩阵（W_Q、W_K、W_V）从同一个 X 投影而来。投影后的维度 d_k = d_model / n_heads（典型值 64）。

📐 **QKV 投影**：Q = X @ W_Q，K = X @ W_K，V = X @ W_V。W 的 shape 均为 (d_model, d_k)。三个投影共享输入 X 但权重组独立——让模型同时学习"找什么""有什么""取什么"三种角色。

💻 **代码　QKV 投影：一次矩阵乘法生成三种角色**




In [ ]:
import numpy as np

np.random.seed(42)
batch, seq_len, d_model, d_k = 2, 5, 512, 64

X = np.random.randn(batch, seq_len, d_model)

# 三个投影矩阵
W_Q = np.random.randn(d_model, d_k) * 0.02
W_K = np.random.randn(d_model, d_k) * 0.02
W_V = np.random.randn(d_model, d_k) * 0.02

Q = X @ W_Q  # (2, 5, 64)
K = X @ W_K  # (2, 5, 64)
V = X @ W_V  # (2, 5, 64)

print(f"X shape: {X.shape}")
print(f"Q shape: {Q.shape} ← (batch, seq_len, d_k)")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nQ[0,2] 是第 0 个样本第 2 个 token 的'我想找什么'向量")
print(f"K[0,3] 是第 0 个样本第 3 个 token 的'我有什么'向量")
print(f"Q[0,2]·K[0,3] = token 2 对 token 3 的注意力得分")




---

### 29.4　缩放点积注意力的逐行拆解

现在进入核心公式。对每一个 query token，计算它与所有 key token 的匹配度，用 softmax 归一化为权重，用权重对 value 加权求和：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

逐行拆解：
1. **`scores = Q @ Kᵀ`**：每一对 (query_i, key_j) 的点积 → 匹配度矩阵 (seq_len, seq_len)
2. **`/ sqrt(d_k)`**：缩放因子。d_k 越大，点积的方差越大 → softmax 越"尖锐"（概率集中在最大值）。除以 √d_k 把方差拉回 1，保持梯度流动
3. **`softmax(scores)`**：沿最后维度归一化 → 每行的注意力权重和为 1
4. **`@ V`**：用权重对 value 加权求和 → 每个 token 的"新表示"是所有 token 的 value 的加权混合

📐 **为什么除 √d_k？** d_k=64 时点积方差 ≈64，softmax 在极端值处梯度趋近于 0（饱和）。除以 √64=8 将方差压回 1，避免饱和。

💻 **代码　缩放点积注意力：完整实现 + 形状追踪**




In [ ]:
import numpy as np

np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

# 1. 得分矩阵
scores = Q @ K.T                      # (4, 4)
print(f"scores shape: {scores.shape}")
print(f"scores:\n{scores.round(2)}\n")

# 2. 缩放
scores_scaled = scores / np.sqrt(d_k)
print(f"After scaling (÷√{d_k}):\n{scores_scaled.round(2)}\n")

# 3. Softmax (稳定版)
scores_shifted = scores_scaled - scores_scaled.max(axis=-1, keepdims=True)
attn_weights = np.exp(scores_shifted)
attn_weights = attn_weights / attn_weights.sum(axis=-1, keepdims=True)
print(f"注意力权重 (每行和=1):\n{attn_weights.round(3)}")
print(f"每行之和: {attn_weights.sum(axis=1).round(3)}\n")

# 4. 加权聚合 Value
output = attn_weights @ V            # (4, 8)
print(f"输出 shape: {output.shape}")

# 验证：scores[i,j] = Q[i] 与 K[j] 的点积
i, j = 2, 1
print(f"\nscores[{i},{j}] = {scores[i,j]:.4f}")
print(f"Q[{i}]·K[{j}]   = {np.dot(Q[i], K[j]):.4f} ✓")




> **关键洞察**：你刚刚实现的这 20 行 NumPy 代码，就是 Transformer 论文中 Scaled Dot-Product Attention 的完整数学逻辑。拆到最底层——`scores[i][j] = Q[i]·K[j]`——就是第 5 章点积的直接应用。整个 LLM 时代建立在这个操作之上。

🔗 **AI 连接**：第 30 章将这个单头注意力复制成 8 份，拼出 Multi-Head Attention。第 32 章的 KV Cache 正是基于"已算过的 K/V 不需要重算"这个简单直觉。

---

### 29.5　因果掩码（Causal Mask）—— 上三角填 `-inf`

在自回归生成时，当前位置不能"偷看"未来的 token——否则就作弊了。**因果掩码（Causal Mask）** 通过将未来位置的注意力得分设为 `-inf`（softmax(-inf)=0）来杜绝这种作弊。

构造方法：生成一个上三角矩阵（不含对角线），上三角区域填 `-inf`，其余填 0。一行 `np.triu` 搞定。

📐 **定义　Causal Mask**：mask[i,j] = 0 if i ≥ j else −inf。确保 token i 只能看到 ≤ i 的位置。

💻 **代码　Causal Mask 的构造与应用**




In [ ]:
import numpy as np

seq_len = 4
# 上三角矩阵（不含对角线）
mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -np.inf
print("Causal Mask:\n", np.where(np.isinf(mask), '-inf', ' 0 '))

# 应用到注意力得分
np.random.seed(42)
scores = np.random.randn(seq_len, seq_len)
scores_masked = scores + mask  # -inf 位置被"屏蔽"

# Softmax(-inf) = 0 — 被屏蔽的位置权重精确为 0
scores_s = scores_masked - scores_masked.max(axis=-1, keepdims=True)
attn = np.exp(scores_s) / np.exp(scores_s).sum(axis=-1, keepdims=True)
print(f"\nMasked attention weights:\n{attn.round(3)}")
print(f"Token 0 只能看自己 (位置 0)")
print(f"Token 2 可以看位置 0,1,2")
print(f"每一行的上三角区域全部为 0 —— '未来'被完美屏蔽")




---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. 打印不同 `d_k` (64 vs 512) 下 scores 的方差，验证缩放因子的作用。
2. 手动构造一个序列，观察掩码后 softmax 是否"看不到未来"。
---
> 预览：下一章——**多头架构与 Block 组装**，从单头到完整层。
